<a href="https://colab.research.google.com/github/kubayishaun-droid/Baardman-Projects/blob/main/neural_network_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project: Slogan Classifier and Generator

In this capstone project you will train a Long Short-Term Memory (LSTM) model to generate slogans for businesses based on their industry, and also train a classifier to predict the industry based on a given slogan.

##Libraries
We recommend running this notebook using [Google Colab](https://colab.google/) however if you choose to use your local machine you will need to install spaCy before starting.

To install spaCy, refer to the installation instructions provided on the spaCy [website](https://spacy.io/usage). Note you may need to install an older version of Python that is compatible with spaCy. You can create a virtual environment for this project to install the specific version of Python that you need.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
import spacy # available on Google Colab
from sklearn.model_selection import train_test_split

## Loading and viewing the dataset

- Load the slogan dataset into a variable called data.
- Extract relevant columns in a variable called df.
- Handle missing values.

Do **not** change the column names.

If you are using Google Colab you will need mount your Google Drive as follows:  
`from google.colab import drive`  
`drive.mount('/content/drive')`  

The path you use when loading your data will look something like this if you are using your Google Drive:  
"/content/drive/MyDrive/Colab Notebooks/slogan-valid.csv"

In [3]:
# Load data from your local machine or from your Google Drive
# Name your pandas DataFrame 'df'

data = pd.read_csv('/content/slogan-valid.csv')

data.head()

print(data.columns)

Index(['desc', 'output', 'type', 'company', 'industry', 'url', 'alias',
       'desc_masked', 'output_masked', 'ent_dict', 'unsupported', 'first_pos'],
      dtype='object')


In [4]:
df = data[['output', 'industry', 'company']].copy()

df = df.dropna()

df.head()

,output,industry,company
0,Taking Care of Small Business Technology,computer hardware,eftpos warehouse
1,Build World-Class Recreation Programs,"health, wellness and fitness",welbi
2,Most Powerful Lead Generation Software for Mar...,internet,optinmonster
3,Hire quality freelancers for your job,internet,twine.fm
4,"Financial Advisers Norwich, Norfolk",financial services,mcb financial services ltd


## Data Preprocessing

Since we are working with textual data, we need software that understands natural language. For this, we'll use a library for processing text called **spaCy**. Using spaCy, we'll break the text into smaller units called tokens that are easier for the machine to process. This process is called **tokenisation**. We'll also convert all text to lowercase and remove punctuation because this information is not necessary for our models. Run the code below, and your dataframe (df) will gain a new column called **'processed_slogan'** which contains the preprocessed text.




In [5]:
# Load spaCy model for text processing
nlp = spacy.load("en_core_web_sm")

# Define text preprocessing function
def preprocess_text(text):
    text_lower = text.lower()
    doc = nlp(text_lower)

    processed_tokens = []

    for token in doc:
        if not token.is_punct:
            processed_tokens.append(token.text)

    return " ".join(processed_tokens)

df["processed_slogan"] = df["output"].apply(preprocess_text)

df.head()

,output,industry,company,processed_slogan
0,Taking Care of Small Business Technology,computer hardware,eftpos warehouse,taking care of small business technology
1,Build World-Class Recreation Programs,"health, wellness and fitness",welbi,build world class recreation programs
2,Most Powerful Lead Generation Software for Mar...,internet,optinmonster,most powerful lead generation software for mar...
3,Hire quality freelancers for your job,internet,twine.fm,hire quality freelancers for your job
4,"Financial Advisers Norwich, Norfolk",financial services,mcb financial services ltd,financial advisers norwich norfolk


We want our model to generate **industry-specific** slogans. If we use the 'processed_slogan' column as it is, we'll be leaving out crucial context - the industries of the companies behind those slogans. To fix this, we'll create a new **'modified_slogan'** column that adds the industry name to the front of processed slogan.  

For example:  

> industry = 'computer hardware'  
processed_slogan = 'taking care of small business technology'  
modified_slogan = 'computer hardware taking care of small business technology'

Write code in the cell below to achieve this.

In [6]:
df['modified_slogan'] = df['industry'] + ' ' + df['processed_slogan']

Now we need to get data to train our model. We have textual data which we will need to represent numerically for our model to learn from it.  
The code below does the following:
1. Tokenizes a dataset of slogans.
2. Converts words to numerical indices.
3. Creates input sequences using the numerical indices.  

Here's how it works. From the 'modified_slogan' column, we take the slogan "computer hardware taking care of small business technology". The tokenisation process will convert words into their corresponding indices:  

<center>

| Word         | Token Index |
|-------------|-------|
| "computer"  | 1     |
| "hardware"  | 2     |
| "taking"    | 3     |
| "care"      | 4     |
| "of"        | 5     |
| "small"     | 6     |
| "business"  | 7     |
| "technology"| 8     |

</center>

So the tokenized list is:

<center>
[1, 2, 3, 4, 5, 6, 7, 8]
</center>

When creating input sequences for training, the loop generates progressively longer sequences.

<center>

| Token Index Sequence               | Corresponding Slogan                                 |
|------------------------------|-----------------------------------------------------|
| [1, 2]                       | "computer hardware"                                |
| [1, 2, 3]                    | "computer hardware taking"                        |
| [1, 2, 3, 4]                 | "computer hardware taking care"                   |
| [1, 2, 3, 4, 5]              | "computer hardware taking care of"                |
| [1, 2, 3, 4, 5, 6]           | "computer hardware taking care of small"          |
| [1, 2, 3, 4, 5, 6, 7]        | "computer hardware taking care of small business" |
| [1, 2, 3, 4, 5, 6, 7, 8]     | "computer hardware taking care of small business technology" |

</center>

Instead of training the model on only **complete slogans**, we provide partial phrases which will help the model learn how words connect over time. This will make it better at predicting the next word when generating slogans.  

Run the cell block below to generate the input sequences. Be sure to read the comments to understand what the code is doing.


In [9]:
'''** Clean up comments'''

# Tokenizer to convert words into numerical values tokens
tokenizer = Tokenizer()

# Tokenizer learns words in dataset
tokenizer.fit_on_texts(df["modified_slogan"])

# Total number of unique words in learned vocabulary
total_words = len(tokenizer.word_index) + 1

# Dictionary mapping words to its numerical index: index based on frequency i.e., more freq => lower index
tokenizer.word_index

# Creating input sequences
# Initialise list to store the input sequences
input_sequences = []

# Iterate over processed slogans
for line in df["modified_slogan"]:

    # Convert slogans to token sequences
    token_list = tokenizer.texts_to_sequences([line])[0] # returns list containing list of words indices; extracting inner list [0]

    # token_list is a list of tokenized word INDICES
    # Building list of progressively longer input sequences for better training
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

The input sequences created above are of **varying lengths**, which will be a problem when training our LSTM model. LSTMs require input sequences of **equal length**. So, we need to **pad** shorter sequences by **prepending zeros** until they match the length of the longest sequence.  

For example, if the longest sequence has **10 tokens**, our padded sequences will look like this:

<center>

| Input Sequence                     | Padded Sequence                         |
|-------------------------------------|-----------------------------------------|
| [1, 2]                              | [0, 0, 0, 0, 0, 0, 0, 0, 1, 2]         |
| [1, 2, 3]                           | [0, 0, 0, 0, 0, 0, 0, 1, 2, 3]         |
| [1, 2, 3, 4]                        | [0, 0, 0, 0, 0, 0, 1, 2, 3, 4]         |
| [1, 2, 3, 4, 5]                     | [0, 0, 0, 0, 0, 1, 2, 3, 4, 5]         |
| [1, 2, 3, 4, 5, 6]                  | [0, 0, 0, 0, 1, 2, 3, 4, 5, 6]         |
| [1, 2, 3, 4, 5, 6, 7]               | [0, 0, 0, 1, 2, 3, 4, 5, 6, 7]         |
| [1, 2, 3, 4, 5, 6, 7, 8]            | [0, 0, 1, 2, 3, 4, 5, 6, 7, 8]         |

</center>

In the cell below, write code that **finds the length of the longest sequence** in **input_sequences** and stores this value in a variable named **max_seq_len**.


In [10]:
max_seq_len = max(len(seq) for seq in input_sequences)

Run the cell below to pad the input sequences so they are all the same length as **max_seq_length**.

In [11]:
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre")

## Training Data for Slogan Generator

The input sequences generated will be used as our training data. Our LSTM needs to learn how to predict the **next word** in a sequence.  

The inputs for our model will be the input sequences **excluding the last token index** and the outputs will be the **last token index**.  

As an example, let us use the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7, 8] and say it corresponds to the slogan "computer hardware taking care of small business technology". When training the model:

> Our input **x** will be the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7] corresponding to "computer hardware taking care of small".  
> Our output **y** will be [8] which corresponds to "business".  

In the code cell below, use `input_sequences` to create the following two variables:
1. **X_gen** which contains the input sequences excluding the last token index.
2. **y_gen** which contains the last token index of the input sequence.

In [12]:
X_gen = input_sequences[:, :-1]
y_gen = input_sequences[:, -1]

The model will output the next word of a sequence over a probability distribution. We need to encode our output variable for this to be possible.

In the code cell below, write code that will apply one-hot encoding to **y_gen** using `tf.keras.utils.to_categorical()`. **Maintain the same variable name**.  

*Hint: set the `num_classes` (number of classes) parameter to the total number of unique words in the learned vocabulary. You can access this value through a variable that was created when generating input sequences earlier.*

In [14]:
print(tf.__version__)
print(tf.__file__)
print(hasattr(tf, "keras"))

2.21.0
/usr/local/lib/python3.13/dist-packages/tensorflow/__init__.py
True


In [16]:
import keras
print(keras.__version__)

3.13.2


In [18]:
y_gen = tf.keras.utils.to_categorical(y_gen, num_classes=total_words)

## Slogan Generator Architecture

In the code cell that follows, configure the LSTM following these steps:

1. Create a sequential model using `tf.keras.models.Sequential()`. This model will have an embedding layer, two LSTM layers, and a dense output layer.
2. Add an embedding layer that converts words into dense vector representations. This layer should:
> *   Have `total_words`as the vocabulary size.
> *   Use 100 as an embedding dimension.
> *   Takes an input length of `max_seq_len - 1` (excludes the target word).
3. Add two LSTM layers.
> *   The first LSTM layer should have 150 **and** set `return_sequences` to `True`.
> *   The second LSTM layer should have 100 units.
4. Add a dense output layer which:
> *   Uses `total_words` as the number of units (one for each word in the vocabulary).
> *   Uses a softmax activation function.
5. Use `Sequential` to put everything together in the correct order to complete the architecture of the LSTM model called **gen_model**.


In [19]:
gen_model = keras.Sequential([
    keras.layers.Input(shape=(max_seq_len - 1,)),
    keras.layers.Embedding(total_words, 100),
    keras.layers.LSTM(150, return_sequences=True),
    keras.layers.LSTM(100),
    keras.layers.Dense(total_words, activation="softmax")
])

In the code cell below, compile `gen_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.


In [21]:
gen_model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

## Slogan Generation

In the code cell below, fit the compiled model on the inputs and outputs, setting the **number of epochs to 50**.

In [22]:
gen_model.fit(X_gen, y_gen, epochs=50)

Epoch 1/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 47s 41ms/step - accuracy: 0.0694 - loss: 7.0370
Epoch 2/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 83s 42ms/step - accuracy: 0.1059 - loss: 6.3596
Epoch 3/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 45s 41ms/step - accuracy: 0.1445 - loss: 6.0252
Epoch 4/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 44s 41ms/step - accuracy: 0.1731 - loss: 5.7669
Epoch 5/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 46s 42ms/step - accuracy: 0.1953 - loss: 5.5444
Epoch 6/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 79s 40ms/step - accuracy: 0.2109 - loss: 5.3335
Epoch 7/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 81s 39ms/step - accuracy: 0.2243 - loss: 5.1267
Epoch 8/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 83s 41ms/step - accuracy: 0.2373 - loss: 4.9294
Epoch 9/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 45s 41ms/step - accuracy: 0.2461 - loss: 4.7459
Epoch 10/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 45s 41ms/step - accuracy: 0.2303 - loss: 4.8687
Epoch 11/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 43s 39ms/step - accuracy: 0.2161 - loss: 4.7434
Epoch 12

We will now define a function called `generate_slogan` which will generate a slogan by predicting one word at a time based on a given starting phrase (the `seed_text`). This function will do this using our trained model, `gen_model`.

Here is a breakdown of how the algorithm works:  

Let us assume the dictionary mapping words to unique indices, `tokenizer.word_index`, looks like this:

> `{'computer': 1, 'hardware': 2, 'taking': 3, 'care': 4, 'of': 5}`

If the model's predicted index for the next word is 3 (`predicted_index = 3`), the loop will:

> Check 'computer' (index 1) → No match  
> Check 'hardware' (index 2) → No match  
> Check 'taking' (index 3) → Match found!  
> Assign output_word = "taking" and exit the loop.  

The `output_word` will be appended to the `seed_text`, and the process will continue to add words to the `seed_text` until we have reached the maximum number of words **or** an invalid prediction occurs.  

Carefully follow the code below and complete the missing parts as guided by the comments.

In [70]:
def generate_slogan(seed_text, max_words=20):
    for _ in range(max_words):

        # Tokenising and padding seed_text
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding="pre")

        # Use your trained model (gen_model) on token_list to predict the probability distribution of the next word over the vocabulary
        predictions = gen_model.predict(token_list, verbose=0)

        # From the predicted probabilities (stored in the variable predictions), identify the word index with the highest probability using np.argmax()
        predicted_index = predictions.argmax(axis=-1)[0]

        output_word = None

        # Searching for the word that corresponds to the predicted index
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # If no valid word is found, algorithm stops
        if output_word is None:
            break # out of main loop

        # If valid a word is found, we append the predicted word to seed_text and continue the process until max_words is reached or an invalid prediction occurs
        # Complete the code below to add a word to seed_text

        seed_text += " " + output_word

    return seed_text

## Training Data for Slogan Classifier

We will now prepare the data we will use to train our classifier. For our classifier, the inputs will come from the `processed_slogans` column of our DataFrame, `df`. The outputs will be the different industry categories under the `industry` column.

In the code cell below, extract the unique values from the `industry` column in the DataFrame and store these in a variable called **industries**.

In [39]:
industry_counts = df["industry"].value_counts()

df = df[df["industry"].map(industry_counts) >= 2].copy()

In [40]:
industries = df["industry"].unique()

Create a dictionary called `industry_to_index` where each unique industry is mapped to a unique index starting from 0.

*Hint: Use the `enumerate()` function.*

In [41]:
industry_to_index = {industry: index for index, industry in enumerate(industries)}

Create a new column `industry_index` in your DataFrame by mapping the `industry` column to the indices using the `industry_to_index` dictionary.

*Hint: Use the  `map()` function.*

In [42]:
df["industry_index"] = df["industry"].map(industry_to_index)

Split the DataFrame `df` into training and testing sets, setting aside 20% of the data for the test set. Be sure to set the parameter `stratify=df["industry_index"]`. This ensures that both sets have the same proportion of each class (industry) as in the original dataset, resulting in balanced datasets. Call the training DataFrame `df_train` and the testing DataFrame `df_test`.

In [37]:
df["industry"].value_counts()

,count
industry,
information technology and services,452
marketing and advertising,267
computer software,257
construction,195
internet,187
...,...
tobacco,1
libraries,1
judiciary,1


In [44]:
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["industry_index"],
    random_state=42

)



Our classifier will use padded slogan sequences as inputs, similar to input sequences used for the slogan generator. The difference is we will not use sequences that get progressively longer, but instead we will use **complete slogans**. This is because our classifier does not need to learn how to predict what word comes next. It needs the full context of a slogan to learn how to accurately predict the industry.  

The next steps will walk you through how to create these sequences.  

We previously created and fitted a `Tokenizer` object called `tokenizer` while preparing data for the slogan generator. Now, we will reuse it to convert words into numerical indices.  

In the code cell below, use the `texts_to_sequences()` **method** of `tokenizer` to transform the `processed_slogan` column in **both** the `df_train` and `df_test` DataFrames into sequences of numerical indices. Store the results in variables named `X_train` and `X_test`.


In [46]:
X_train = tokenizer.texts_to_sequences(df_train["processed_slogan"])
X_test = tokenizer.texts_to_sequences(df_test["processed_slogan"])

The slogan sequences are of varying lengths. We will need to pad them the same way we did to the input sequences for the slogan generator. The `pad_sequences()` function can ensure the sequences in `slogan_sequences` have the same length.  

In the code cell below, use the `pad_sequences()` function to standardise the `slogan_sequences` lengths. Set the `maxlen` parameter to `max_seq_len`, the `padding` parameter to 0, and assign the resulting padded sequences to the same variables, `X_train` and `X_test`.

In [50]:
X_train = pad_sequences(X_train, maxlen=max_seq_len, padding="post")
X_test = pad_sequences(X_test, maxlen=max_seq_len, padding="post")

We have successfully created training and testing inputs for our model. Now, we will create the outputs - industry categories.

 In the code cell that follows, use `tf.keras.utils.to_categorical()` to apply one-hot encoding to the `industry_index` column of **both** `df_train` and `df_test` DataFrames. Assign the results to a variables named `y_train` and `y_test`.

 *Hint: set the `num_classes` parameter to the total number of industries in the DataFrame. The `industries` variable can be used to find this value.*

In [54]:
y_train = tf.keras.utils.to_categorical(df_train["industry_index"], num_classes=len(industries))
y_test = tf.keras.utils.to_categorical(df_test["industry_index"], num_classes=len(industries))

## Slogan Classifier Architecture

Configure the LSTM classifier following these steps:  


1. Create a Sequential model:  
   Use `tf.keras.models.Sequential()` to create a sequential model. This model will consist of an embedding layer, two LSTM layers, and a dense output layer.

2. Add an embedding layer which will convert words into dense vector representations. Configure this layer with:
   > * `total_words` as the vocabulary size.
   > * 100 as the embedding dimension.
   > * `max_seq_len` as the `input_length` (this is the length of the slogans).

3. Add the first LSTM layer. Configure it with:
   > * 150 units.
   > * Set `return_sequences` to `True` to ensure the layer outputs sequences for the next LSTM layer.

4. Add the second LSTM layer which will process the output from the previous LSTM layer. Configure it with:
   > * 100 units.
   > * No need to set `return_sequences` here (it is the final LSTM layer).

5. Add the dense output layer which will classify the data into industries. Configure it with:
   > * The number of unique industries as the number of units.
   > * The `softmax` activation function to get probabilities for each class (industry).

6. Use `Sequential` to arrange all layers in the correct order and complete the architecture of the LSTM model called **class_model**.


In [56]:
class_model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(total_words, 100, input_length=max_seq_len),
    tf.keras.layers.LSTM(150, return_sequences=True),
    tf.keras.layers.LSTM(100),
    tf.keras.layers.Dense(len(industries), activation="softmax")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In the code cell below, compile `class_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.

In [62]:
class_model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [63]:
history = class_model.fit(
    X_train,
    y_train,
    epochs=50
)

Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.0761 - loss: 4.3828
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.0854 - loss: 4.2695
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.0936 - loss: 4.0490
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.1140 - loss: 3.8486
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.1243 - loss: 3.6913
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.1409 - loss: 3.5376
Epoch 7/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.1589 - loss: 3.3966
Epoch 8/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.1880 - loss: 3.2107
Epoch 9/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.2303 - loss: 2.9939
Epoch 10/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.2743 - loss: 2.8152
Epoch 11/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.3052 - loss: 2.6434
Epoch 12/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 

## Slogan Classification & Evaluation

In the code cell that follows, fit the compiled model on the inputs and outputs, setting **the number of epochs to 50**.

Evaluate the model using the testing set. Add a comment on the model's performance.

In [64]:
test_loss, test_accuracy =class_model.evaluate(X_test, y_test)


34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.1620 - loss: 7.0862


The model achieved high accuracy on the traning data but only 16.20% accuracy on the testing data. This suggests overfitting, meaning the model struggles to generalise to unseen slogans.

We will now define a function called `classify_slogan` which takes a slogan as input and predicts the industry it belongs to using the trained model, `class_model`.  

Carefully follow the code below and complete the missing parts (indicated by ellipses) as guided by the comments.

In [72]:
def classify_slogan(slogan):
    # Use the preprocess_text to clean the input slogan (we defined this function in the preprocessing section)
    slogan = preprocess_text(slogan)

    # Converting the slogan to a sequence of indices
    sequence = tokenizer.texts_to_sequences([slogan])

    # Pad the sequence using the pad_sequences() function
    padded_sequence = pad_sequences(sequence, maxlen=max_seq_len, padding="post")

    # Pass padded_sequence into the class_model to get the predicted probabilities for each indistry
    prediction = class_model.predict(padded_sequence)

    # Use np.argmax() to get the index of the industry with the highest probability.
    # Note: np.argmax() returns a NumPy array, specifically a one-element array in this context.
    # Extract the single value from this array and assign it to predicted_index.
    predicted_index = np.argmax(prediction, axis=1)[0]

    # Return the predicted industry
    return industries[predicted_index]

## Combining the two models

Run the code cell below to combine the two models: we will first generate a slogan for a company in the "internet" industry, then pass the generated slogan to the slogan classifier to see if it correctly classifies it as internet.

In [73]:
industry = "internet"
generated_slogan = generate_slogan(industry)
predicted_industry = classify_slogan(generated_slogan)

print(f"Generated Slogan: {generated_slogan}")
print(f"Predicted Industry: {predicted_industry}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step
Generated Slogan: internet web design agency digital marketing specialists in nyc philly area delivery maintenance texas in paris banners of canada online the
Predicted Industry: business supplies and equipment


Compare the results and comment on any differences you notice between the generated slogans and the classifier’s predictions in the markdown cell below.


The generated slogan relates to the internet industry, but the classifier predicted business supplies and equipment. This shows that the classifier does not always correctly identify the industry, possibly due to limitations in its training data or model accuracy.